In [5]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import copy

In [7]:
df = pd.read_csv(r"C:\Users\SRINIDHI\OneDrive\Desktop\FL-Sem6\diabetes_prediction_dataset.csv")

# Convert categorical → numeric
df = pd.get_dummies(df, drop_first=True)

X = df.drop("diabetes", axis=1).values
y = df["diabetes"].values

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [8]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.long)

input_dim = X_train.shape[1]

In [9]:
class DiabetesModel(nn.Module):
    def _init_(self):
        super()._init_()
        self.fc1 = nn.Linear(input_dim, 64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, 2)

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

In [10]:
num_clients = 5
client_size = len(X_train) // num_clients

clients = []
for i in range(num_clients):
    start = i * client_size
    end = (i + 1) * client_size if i != num_clients - 1 else len(X_train)
    clients.append((X_train[start:end], y_train[start:end]))

In [11]:
def train_local(model, X, y, epochs=1):
    model.train()
    optimizer = optim.SGD(model.parameters(), lr=0.01)
    loss_fn = nn.CrossEntropyLoss()

    for _ in range(epochs):
        optimizer.zero_grad()
        out = model(X)
        loss = loss_fn(out, y)
        loss.backward()
        optimizer.step()

    return model.state_dict()

In [12]:
def evaluate(model):
    model.eval()
    with torch.no_grad():
        out = model(X_test)
        preds = torch.argmax(out, dim=1)
        acc = (preds == y_test).float().mean().item() * 100
    return acc

In [13]:
def weighted_fedavg(local_weights, data_sizes):
    global_weights = copy.deepcopy(local_weights[0])
    total_data = sum(data_sizes)

    for key in global_weights.keys():
        global_weights[key] = sum(
            (data_sizes[i] / total_data) * local_weights[i][key]
            for i in range(len(local_weights))
        )

    return global_weights

In [15]:
rounds = 20





global_model = DiabetesModel()
accuracy_history = []

for r in range(rounds):

    print(f"\n================ ROUND {r+1} ================")

    local_weights = []
    data_sizes = []

    # ----- GLOBAL MODEL DISTRIBUTION -----
    print("📤 Server → Clients : Distributing global model")

    # ----- CLIENT TRAINING -----
    for i, (Xc, yc) in enumerate(clients):

        print(f"👤 Client {i+1} training...")

        local_model = DiabetesModel()
        local_model.load_state_dict(global_model.state_dict())

        weights = train_local(local_model, Xc, yc, epochs=1)

        local_weights.append(weights)
        data_sizes.append(len(Xc))

    # ----- CLIENT → SERVER UPDATES -----
    print("📥 Clients → Server : Sending model updates")

    # ----- SERVER AGGREGATION -----
    print("🧠 Server : Aggregating using Weighted FedAvg")
    global_weights = weighted_fedavg(local_weights, data_sizes)

    # ----- SERVER UPDATE GLOBAL MODEL -----
    global_model.load_state_dict(global_weights)

    # ----- REDISTRIBUTION -----
    print("📤 Server → Clients : Redistributing updated global model")

    # ----- EVALUATION -----
    acc = evaluate(global_model)
    accuracy_history.append(acc)

    print(f"✅ Global Model Accuracy: {acc:.2f}%")

# =========================
# 8. PLOT ACCURACY
# =========================
plt.plot(range(1, rounds + 1), accuracy_history, marker='o')
plt.xlabel("Communication Rounds")
plt.ylabel("Global Accuracy (%)")
plt.title("Federated Learning with Update & Redistribution")
plt.grid(True)
plt.show()


================ ROUND 1 ================
📤 Server → Clients : Distributing global model
👤 Client 1 training...


ValueError: optimizer got an empty parameter list